# Robot arm, live testing 03.03.2026

### Connect to the simulator / to the robot

In [1]:
import sys
from pathlib import Path
import json
import numpy as np

# Make sure ur3e-control is on the path
sys.path.insert(0, str(Path.cwd().parents[1]))

from URBasic import Joint6D, TCP6D
from URBasic.waypoint6d import TCP6DDescriptor

In [2]:
file_path = "trapezoid_letters-trapezoid_letters-trace.json"
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)
print(data["generated_at"])

2026-02-26T14:37:43.905785


------ simulation only ----------

In [3]:
from duckify_simulation.duckify_sim import DuckifySim as ISCoin

SIMULATION = True

# # Connect to the simulator
iscoin = ISCoin()
robot = iscoin.robot_control
print("Connected to simulator")

DuckifySim connected to container 'iscoin_simulator'
Connected to simulator


----------- REAL ROBOT ONLY ----------------

In [4]:
#from URBasic import ISCoin

In [5]:
# Create a new ISCoin object
# UR3e1 IP (closest to window): 10.30.5.158
# UR3e2 IP: 10.30.5.159
if not SIMULATION:
    iscoin = ISCoin(host="10.30.5.158", opened_gripper_size_mm=40)

In [6]:
robot_arm = iscoin.robot_control

## Default home position

This joint position is equivalent to this one:
```
home_tcp = TCP6D.createFromMetersRadians(
      0.0,      # x
     -0.35,     # y
      0.20,     # z
      0.0,      # rx
      3.14,      # ry
      0.0       # rz
  )
```



In [7]:
# Go to home position
home = Joint6D.createFromRadians(1.1859, -1.4452, 1.2389, -1.3639, -1.5693, -0.3849)
robot_arm.movej(home)

tcp_home = robot_arm.get_actual_tcp_pose()
print(f"Home TCP: {tcp_home}")

movej sent (duration=3s)
Movement OK — target reached
Home TCP: TCP6D([-0.0002415726356282012, -0.3499990478240247, 0.3450026529801197, -5.0664473233373674e-06, 3.139981903278723, -1.4319116951655066e-05])


In [8]:
# target_tcp = TCP6D.createFromMetersRadians(
#       0.0,      # x
#      -0.35,     # y
#       0.20,     # z
#       0.0,      # rx
#       3.14,      # ry
#       0.0       # rz
#   )

In [9]:
# current_joints = robot.get_actual_joint_positions()
# target_joints = robot.get_inverse_kin(target_tcp, qnear=current_joints)

## Set TCP

In [10]:
if not SIMULATION:
    from src.calibration import collect_data
    NUM_MEASURES = 20

    tcps = collect_data(robot_arm, NUM_MEASURES)
else:
    tcps = [
        [0.0024946337740892888, -0.3594913915339868, 0.06033167700558813, -0.62069755529755, -2.369727486212462, -0.13752393611863362],
        [-0.07482225018547103, -0.35685269130655106, 0.07009337653443365, 1.9372057315922462, 2.190232377619723, 0.6719840297817087],
        [-0.08776793211801745, -0.2979812165503483, 0.051154992555969794, 2.3781388566898687, 0.2623998383341909, 0.7178051284770465],
        [-0.0006933205535983589, -0.340924257935631, 0.06551204074775754, -2.879088789274111, -0.4716877213448937, 0.8276834472055694],
        [-0.04094004360899719, -0.3437262194580261, 0.0786140415986546, -3.087426800813378, 0.2627406174702693, 0.06421242457662117],
        [0.026621421400990636, -0.3469050147213359, 0.033956680032955974, -2.5014091646201537, -0.22918589994059693, 1.5168173293250897],
    ]

In [11]:
from src.calibration import get_tcp_offset, validate_calibration
tcp_offset = get_tcp_offset(tcps)
robot_arm.set_tcp(tcp_offset)
if not SIMULATION:
    robot_arm.freedrive_mode()
    input()
    robot_arm.end_freedrive_mode()
else:
    robot_arm.set_tcp(tcp_offset)
    
motion_val_cal = validate_calibration(robot_arm)

TCP consistency (mm):    	 0.5873600179479618
Flange motion (cm):      	 10.961315598701008
Rotation variation (deg):	 36.9973854515488
TCP in flange frame: [-0.00072928 -0.00040785  0.07966732]  <-
TCP in tool   frame: [-0.04418502 -0.33927468 -0.00117747]
Rotating: rx=0.2, ry=0, rz=0
Rotating: rx=-0.2, ry=0, rz=0
Rotating: rx=0, ry=0.2, rz=0
Rotating: rx=0, ry=-0.2, rz=0
Rotating: rx=0, ry=0, rz=0.2
Rotating: rx=0, ry=0, rz=-0.2


In [12]:
for m in motion_val_cal:
    robot_arm.movel(m)
    break

movej sent (duration=2s)
Movement OK — target reached


In [13]:
def _normal_to_rxyz(n):
    x,y,z = n
    z = -z
    rx = np.arctan2(-y, np.sqrt(x**2 + z**2))
    ry = np.arctan2(x, z)
    rz = 0
    return [rx,ry,rz]

In [14]:
def create_transformation(A, B):
    A = np.asarray(A)[:, :3]
    B = np.asarray(B)[:, :3]
    assert A.shape == B.shape
    N = A.shape[0]

    # Build matrix for affine solve: [A | 1]
    P = np.hstack([A, np.ones((N, 1))])  # Nx4

    # Solve P @ X = B, where X is 4×3 (R^T and t)
    X, _, _, _ = np.linalg.lstsq(P, B, rcond=None)

    # Extract R and t
    R = X[:3, :].T   # 3×3
    t = X[3, :]      # 3

    # Build full 4×4 affine matrix
    T = np.eye(4)
    T[:3, :3] = R
    T[:3, 3] = t

    # Normal transform
    R_normal = np.linalg.inv(R).T
    T_normal = np.eye(4)
    T_normal[:3, :3] = R_normal

    def AtoB(p):
        p = np.asarray(p)
        point = p[:3]
        normal = p[3:]

        # Transform point
        p_new = T @ [*point, 1]

        # Transform normal
        nx,ny,nz = normal
        n_new = T_normal @ [nx,ny,nz, 1]
        n_new /= np.linalg.norm(n_new)
        r_new = _normal_to_rxyz(n_new[:3])

        return [*p_new[:3], *r_new]

    return AtoB


In [15]:
from src.transformation import collect_data as collect_data_tf
p_world = np.array(data["calibration"])
if len(p_world) < 4:
    print("Missing point")
if not SIMULATION:
    p_tcp = collect_data_tf(robot_arm, p_world)
else:
    p_tcp = np.array([
        [-0.03389773, -0.2829937,   0.09458801],# -0.24982057, -3.09150359, -0.18723581],
        [-0.03366482, -0.31970029,  0.09458314],# -0.28493231, -3.12244751, -0.07563827],
        [ 0.02527303, -0.32058601,  0.09427673],#  0.21952408,  3.00241046,  0.09098521],
        [ 0.05483491, -0.26206449,  0.05878293]#,  0.18852376, -2.79599659, -0.21783004]
    ])
    # p_tcp = np.array([
    #     [-0.030, -0.285, 0.180],
    #     [-0.030, -0.285, 0.220],
    #     [ 0.030, -0.285, 0.220],
    #     [ 0.060, -0.25,  0.160]
    # ])

world2tcp = create_transformation(p_world, p_tcp)

In [16]:
faces = [
    [0.0, 0.9999999999999999, 0.0],
    [0.0, 0.4999999425479707, 0.8660254369543807],
    [0.0, 0.500000078148597, -0.8660253586653204],
    [0.7660444804970836, 0.6427875651410451, 0.0],
    [-0.7660444410855507, 0.6427876121098819, 4.820907090824115e-08]
]
p_w = [0,0,0]
for f in faces:
    t = world2tcp([*p_w, *f])
    print(t)

[np.float64(-0.0038623056369459646), np.float64(-0.2991027187550362), np.float64(0.05908430848848012), np.float64(9.969413049399048e-05), np.float64(3.1363953323508764), 0]
[np.float64(-0.0038623056369459646), np.float64(-0.2991027187550362), np.float64(0.05908430848848012), np.float64(1.1252594732526686), np.float64(-3.11531848524934), 0]
[np.float64(-0.0038623056369459646), np.float64(-0.2991027187550362), np.float64(0.05908430848848012), np.float64(-1.0084978700335714), np.float64(3.1125437811899714), 0]
[np.float64(-0.0038623056369459646), np.float64(-0.2991027187550362), np.float64(0.05908430848848012), np.float64(-0.004765418476329431), np.float64(2.271306470917294), 0]
[np.float64(-0.0038623056369459646), np.float64(-0.2991027187550362), np.float64(0.05908430848848012), np.float64(0.004981390379891859), np.float64(-2.2599746781251744), 0]


Generate trajectory

In [17]:
from src.transformation import tcp_trans
traces = data["traces"]
draw_motions = []
move_motions = []
above = [0,0,-0.02,0,0,0]
for trace in traces:
    normal = trace["face"]
    normal[2] = - normal[2]
    t_world = trace["path"]
    motion = []
    buffer_motion = []
    
    # Drawing motion
    for i, p in enumerate(t_world):
        p_w = (*p, *normal)
        x,y,z,rx,ry,rz = world2tcp(p_w)
        
        # Add entry point
        if i == 0:
            x_t,y_t,z_t,rx_t,ry_t,rz_t = tcp_trans([x,y,z,rx,ry,rz], above)
            m = TCP6D.createFromMetersRadians(x_t,y_t,z_t, rx,ry,rz)
            buffer_motion.append(m)

        # Add drawing point
        m = TCP6D.createFromMetersRadians(x,y,z, rx,ry,rz)
        buffer_motion.append(m)

        # Add exit point
        if i == len(t_world)-1:
            x_t,y_t,z_t,rx_t,ry_t,rz_t = tcp_trans([x,y,z,rx,ry,rz], above)
            m = TCP6D.createFromMetersRadians(x_t,y_t,z_t, rx,ry,rz)
            m = TCP6D.createFromMetersRadians(x,y,z, rx,ry,rz)
            buffer_motion.append(m)
        if len(buffer_motion) >= 20:
            motion.append(buffer_motion)
            buffer_motion = []
    if buffer_motion:
        motion.append(buffer_motion)
    if motion:
        draw_motions.append(motion)
        move_motions.append((motion[0][0], motion[-1][-1]))
    #break

In [18]:
move_outside = []
for m in range(len(move_motions)):
    if m == 0:
        move_outside.append([home, move_motions[m][0]])
    else:
        move_outside.append([move_motions[m-1][-1], home, move_motions[m][0]])
move_outside.append([move_motions[-1][-1], home])

### Drawing algorithm live

In [19]:
robot_arm.movej(home)
for k in range(len(move_motions)):
    #py.move(move_outside[k])
    print(data["traces"][k]["face"])
    for m in move_outside[k]:
        if isinstance(m, TCP6D):
            robot_arm.movel(m)
            print(m)
        
        elif isinstance(m, Joint6D):
            robot_arm.movej(m)
            print(robot_arm.get_actual_tcp_pose())
    
    print()
    #for m in draw_motions[k]:
    #    robot_arm.movel_waypoints(m)
print("Return home")
for m in move_outside[-1]:
    if isinstance(m, TCP6D):
        robot_arm.movel(m)
        print(m)
    elif isinstance(m, Joint6D):
        robot_arm.movej(m)
        print(robot_arm.get_actual_tcp_pose())


movej sent (duration=3s)
Movement OK — target reached
[0.0, 0.9999999999999999, 0.0]
movej sent (duration=3s)
Movement OK — target reached
TCP6D([0.0006160330416389668, -0.3504076260633291, 0.2653366127630045, -5.066447313139561e-06, 3.139981903278723, -1.4319116950879414e-05])
movej sent (duration=2s)
Movement OK — target reached
TCP6D([-0.01474048155003873, -0.3073043209420538, 0.11448482215513381, 9.969413049399048e-05, 3.1363953323508764, 0.0])

[0.0, 0.4999999425479707, 0.8660254369543807]
movej sent (duration=2s)
Movement OK — target reached
TCP6D([-0.01466238504748062, -0.3032304581440587, 0.09448563277082955, 9.969413049399048e-05, 3.1363953323508764, 0.0])
movej sent (duration=3s)
Movement OK — target reached
TCP6D([0.0006160330416388914, -0.3504076260633293, 0.2653366127630062, -5.066447319580889e-06, 3.1399819032729015, -1.431911704165739e-05])
movej sent (duration=2s)
Movement OK — target reached
TCP6D([-0.01688430562697292, -0.27742935676494235, 0.10304363607867634, 1.1252